In [ ]:
# ========== 第 3 周练习：Hugging Face Llama 旅行社聊天 ==========
# 理念：用本地/Colab 上的开源 Llama，做一个「带偏见」的旅行助手：
# - 系统提示故意把用户往 Spain（西班牙）引导（演示 prompt 如何塑造行为）
# - 回答后再用同一模型流式翻译成 Hindi（印地语），展示 TextIteratorStreamer
# 运行环境：Google Colab + HF_TOKEN（Secrets）+ 能拉取 gated Llama 模型
# （本格只有说明注释，无执行逻辑）
# 一家 HuggingFace LLAMA 旅行社使用翻译支持，偏向某一特定目的地。


In [ ]:
# ========== 导入：Gradio UI + Transformers 推理 + Colab 密钥 ==========

# Gradio：快速搭聊天界面（ChatInterface）
import gradio as gr
# PyTorch：张量与 GPU/CPU 设备
import torch
# AutoTokenizer：文本↔token；AutoModelForCausalLM：因果语言模型；TextIteratorStreamer：边生成边吐 token
from transformers import AutoTokenizer, AutoModelForCausalLM, TextIteratorStreamer
# Thread：把 model.generate 放到后台线程，主线程可消费 streamer
from threading import Thread
# login：用 Hugging Face token 登录（gated 模型需要）
from huggingface_hub import login
# userdata：从 Colab Secrets 读 HF_TOKEN，避免把密钥写进笔记本
from google.colab import userdata


In [ ]:
# ========== 登录 HF + 加载 Llama 分词器与模型 ==========

# 要加载的 Instruct 模型 id（字符串勿改；需有权访问该 gated repo）
model_name = "meta-llama/Llama-3.2-1B-Instruct"
# 打印进度，方便在 Colab 长下载时确认卡在哪一步
print(f"Loading {model_name}...")

# 下面两行是注释掉的旧写法示例（dotenv / OpenWeather），保留原样不启用
# load_dotenv(覆盖=True)
# OPENWEATHER_API_KEY = os.getenv("OPENWEATHER_API_KEY")
# 从 Colab Secrets 取出名为 HF_TOKEN 的密钥
hf_token = userdata.get('HF_TOKEN')
# 登录 Hugging Face Hub；add_to_git_credential=True 便于后续 git/lfs 拉模型
login(hf_token, add_to_git_credential=True)

# 按 model_name 下载/缓存分词器
tokenizer = AutoTokenizer.from_pretrained(model_name)
# 因果 LM 常无 pad_token：用 eos 顶上，避免 batch/pad 时报错
tokenizer.pad_token = tokenizer.eos_token
# 加载因果语言模型：bfloat16 省显存；device_map="auto" 自动放到可用 GPU
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

# 加载完成提示（英文原样保留，便于对照日志）
print("Model loaded successfully!")


In [ ]:
# ========== 英→印地语流式翻译：生成器 do_translate ==========

def do_translate(msg):
    """
    Translator function to format the response.
    """
    # system prompt：定角色为英→印地语翻译专家（影响行为的英文勿改）
    sys_msg = """
    You are an expert translator who can translate the given English text to Hindi.
    """

    # user prompt：把待译英文 msg 嵌进指令
    usr_msg = f"""
    Translate the given English text to Hindi.\n
    {msg}
    """

    # Chat messages：system + user，供 chat template 使用
    messages = [{"role": "system", "content": sys_msg}, {"role": "user", "content": usr_msg}]

    # apply_chat_template：按 Llama Instruct 格式拼成单条输入字符串；add_generation_prompt 追加助手开头
    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # 再 tokenize 成张量，并搬到与 model 相同的 device
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    # TextIteratorStreamer：generate 时把新 token 解码后放入可迭代队列
    streamer = TextIteratorStreamer(
        tokenizer,
        skip_prompt=True,
        skip_special_tokens=True
    )

    # generation_kwargs：传给 model.generate；dict(inputs, ...) 会把 inputs 里的键展开进去
    generation_kwargs = dict(
        inputs,
        streamer=streamer,
        max_new_tokens=512,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
    )

    # 后台线程跑 generate，避免阻塞主线程读 streamer
    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    # 先立刻 yield：原文 + Hindi 标题，UI 立刻有内容
    partial_response = msg + '\n**Hindi**: '
    yield partial_response
    # 再逐段拼接 streamer 吐出的新文本并 yield（流式刷新）
    for new_text in streamer:
        partial_response += new_text
        yield partial_response



In [ ]:
# ========== Gradio 聊天回调：先 Llama 答旅问，再 yield from 翻译 ==========

def chat_with_llama(message, history):
    """
    Chat function that streams responses from the Llama model.

    Args:
        message: The user's current message
        history: List of [user_message, assistant_message] pairs

    Yields:
        Partial responses as they are generated
    """

    # system：旅行社助手 + 故意偏向 Spain（演示「操纵性」prompt；英文勿改）
    sys_msg = """
      You are a manipulative assistant for a travel company.
      Give short answers.
      You have the motive to make the user to travel to Spain than any other country.
      """
    # 对话历史从 system 开始
    messages = [{"role": "system", "content": sys_msg}]

    # Gradio history 是 [user, assistant] 对；助手侧可能带「\n**Hindi**: …」，喂回模型时只取英文部分
    for user_msg, assistant_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": assistant_msg.split('\n**Hindi**: ')[0]})

    # 追加当前用户输入
    messages.append({"role": "user", "content": message})

    # 同样用 chat template 拼成模型输入字符串
    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # tokenize 并放到模型设备
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    # 无梯度推理，省显存；此处非流式，一次 generate 完整英文回答
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
        )

    # 只解码「新生成」那段：从原 input_ids 长度之后切到结尾
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

    # 把英文回答交给 do_translate，把其流式 yield 原样转发出去
    yield from do_translate(response)



In [ ]:
# ========== 搭建 Gradio ChatInterface 并启动（share 外链） ==========

# 从完整 model id 取出最后一段当显示名，例如 Llama-3.2-1B-Instruct
w_model = model_name.split('/')[-1]
# ChatInterface：把 chat_with_llama 接到聊天 UI；title/description/examples 是给人看的字符串（可保持英文）
demo = gr.ChatInterface(
    fn=chat_with_llama,
    title = f"🦙 {w_model} Chat",
    description = f"Chat with Meta's {w_model} model with streaming responses",
    examples=[
        "What is the capital of France?",
        "I want to travel to America",
        "What are some tips for learning a new language?"
    ],
    theme=gr.themes.Soft()
)

# share=True：生成临时公网链接；debug=True：出错时打印更详细栈
demo.launch(share=True, debug=True)

